# Naive RAG Experiments

Runs the baseline Naive RAG pipeline across different vector databases and embedding models.
This serves as the control experiment for comparison with Hybrid and Multi-Agent approaches.

**Pipeline:** Query → Dense Retrieval (Cosine/MMR) → LLM Generation → Answer  
**Variables:** Vector DB (Chroma, Qdrant, LanceDB), Embedding model (MiniLM, BGE)  
**Evaluation:** RAGAS and DeepEval metrics

In [49]:
import sys
sys.path.append("..")

import os
import time
import json
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
import config
from ast import literal_eval
from deepeval.evaluate import DisplayConfig, AsyncConfig
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric,
)
import deepeval
import instructor
from groq import AsyncGroq

from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

/tmp/ipykernel_1284/2295925951.py:17: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_1284/2295925951.py:17: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextPrecisionWithReference
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_1284/2295925951.py:17: DeprecationWarning: Importing BleuScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.

## Load Vector Stores

Loaders for each vector DB ingested by `ingestion_pipeline.ipynb`. All functions are
read-only — they never re-embed or re-write. Pick whichever store you want to evaluate.

In [6]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    """Load an existing ChromaDB collection from disk.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        db_name: Collection name; defaults to {DEFAULT_EMBEDDING}_pubmed_chroma.
        persist_dir: Override storage path (defaults to vectorstores/{db_name}).

    Returns:
        Chroma vector store instance.
    """
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )


def load_faiss(embeddings, index_name=None, path=None):
    """Load an existing FAISS index from disk.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        index_name: Subdirectory name under vectorstores/; defaults to {DEFAULT_EMBEDDING}_pubmed_faiss.
        path: Override storage path.

    Returns:
        FAISS vector store instance.
    """
    from langchain_community.vectorstores import FAISS

    index_name = index_name or f"{config.DEFAULT_EMBEDDING}_pubmed_faiss"
    save_path = path or str(config.VECTORSTORE_DIR / index_name)
    print(f"Loading FAISS index from {save_path}")
    return FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)


def load_qdrant(embeddings, collection_name=None, path=None):
    """Load an existing local Qdrant collection.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        collection_name: Collection name; defaults to {DEFAULT_EMBEDDING}_pubmed_qdrant.
        path: Override storage path (defaults to vectorstores/{collection_name}).

    Returns:
        QdrantVectorStore instance.
    """
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient

    collection_name = collection_name or f"{config.DEFAULT_EMBEDDING}_pubmed_qdrant"
    path = path or str(config.VECTORSTORE_DIR / collection_name)
    print(f"Loading Qdrant collection '{collection_name}' from {path}")
    client = QdrantClient(path=path)
    return QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )


def load_lancedb(embeddings, table_name=None, path=None):
    """Load an existing LanceDB table as a LangChain-compatible vector store.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        table_name: Table name; defaults to {DEFAULT_EMBEDDING}_pubmed_lance.
        path: Override storage path (defaults to vectorstores/lancedb).

    Returns:
        LanceDB vector store instance.
    """
    import lancedb
    from langchain_community.vectorstores import LanceDB as LanceDBStore

    table_name = table_name or f"{config.DEFAULT_EMBEDDING}_pubmed_lance"
    path = path or str(config.VECTORSTORE_DIR / "lancedb")
    print(f"Loading LanceDB table '{table_name}' from {path}")
    db = lancedb.connect(path)
    table = db.open_table(table_name)
    return LanceDBStore(connection=table, embedding=embeddings, text_key="text")

In [7]:
def get_cosine_retriever(vector_store, k=None):
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})


def get_mmr_retriever(vector_store, k=None, fetch_k=None, lambda_mult=None):
    k = k or config.TOP_K
    fetch_k = fetch_k or config.MMR_FETCH_K
    lambda_mult = lambda_mult or config.MMR_LAMBDA_MULT
    return vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k, "fetch_k": fetch_k, "lambda_mult": lambda_mult},
    )

## LLM & RAG Chain

In [8]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


def is_rate_limit_error(e):
    msg = str(e).lower()
    return any(kw in msg for kw in [
        "rate_limit", "rate limit", "429", "too many requests",
        "tokens per", "token limit", "exceeded",
    ])


RAG_PROMPT_TEMPLATE = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)


def build_naive_rag_chain(llm):
    return RAG_PROMPT | llm


def run_rag(retriever, df, key_rotator, delay=20):
    """Run the RAG pipeline sequentially, rotating API keys on rate/token limit errors.

    Single-threaded fallback — use run_rag_parallel when multiple API keys are available.
    Adds retrieved_contexts and generated_answer as new columns to a copy of df.

    Args:
        retriever: LangChain retriever instance.
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator instance managing API key cycling.
        delay: Seconds to sleep between successful requests (default 20).

    Returns:
        a copy of df with two new columns:
            retrieved_contexts (list of retrieved chunk strings) and generated_answer.
    """
    result_df = df.copy().reset_index(drop=True)
    retrieved_contexts_list = [None] * len(df)
    generated_answer_list = [None] * len(df)

    chain = build_naive_rag_chain(key_rotator.get_llm())

    for idx, (_, row) in enumerate(df.iterrows()):
        question = row["question"]
        rotations_attempted = 0

        while True:
            try:
                contexts = retriever.invoke(question)
                result = chain.invoke({"context": contexts, "question": question})
                retrieved_contexts_list[idx] = [doc.page_content for doc in contexts]
                generated_answer_list[idx] = result.content
                time.sleep(delay)
                break
            except Exception as e:
                if is_rate_limit_error(e) and rotations_attempted < len(key_rotator.api_keys):
                    print(f"[{idx + 1}/{len(df)}] Rate limit on key {key_rotator.current_idx}: {str(e)[:100]}")
                    key_rotator.rotate()
                    chain = build_naive_rag_chain(key_rotator.get_llm())
                    rotations_attempted += 1
                    time.sleep(10)
                else:
                    print(f"[{idx + 1}/{len(df)}] Skipping '{question[:50]}...': {e}")
                    break

    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["generated_answer"] = generated_answer_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"\nCompleted {completed}/{len(df)} questions")
    return result_df


def _run_slice(retriever, slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice of the evaluation set using a single dedicated API key.

    Called in its own thread by run_rag_parallel. Rows are processed sequentially
    with `delay` seconds between requests to stay within the key's daily quota.
    Adds retrieved_contexts and generated_answer as new columns to a copy of slice_df.

    Args:
        retriever: LangChain retriever (read-only, safe to call from multiple threads).
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with two new columns: retrieved_contexts and generated_answer.
    """
    chain = build_naive_rag_chain(ChatGroq(model=model, api_key=api_key))
    result_df = slice_df.copy().reset_index(drop=True)
    retrieved_contexts_list = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            contexts = retriever.invoke(question)
            result = chain.invoke({"context": contexts, "question": question})
            retrieved_contexts_list[row_idx] = [doc.page_content for doc in contexts]
            generated_answer_list[row_idx] = result.content
        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["generated_answer"] = generated_answer_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_rag_parallel(retriever, df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next rows_per_key rows, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    ThreadPoolExecutor is used (not asyncio) because: (1) network I/O releases the GIL
    so threads genuinely run concurrently, (2) Jupyter/Colab already have a running event
    loop so asyncio.run() raises RuntimeError, and (3) ChatGroq.invoke() is synchronous.

    Args:
        retriever: LangChain retriever instance.
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        a copy of df with two new columns:
            retrieved_contexts and generated_answer — in original row order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(
            f"Warning: {len(df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} rows = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                retriever, s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback = s.copy().reset_index(drop=True)
                fallback["retrieved_contexts"] = [None] * len(s)
                fallback["generated_answer"] = [None] * len(s)
                ordered_results[idx] = fallback

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    return final_df

## Evaluation Functions

In [47]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    """Return a minimal DataFrame with question_index and metric score."""
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame.

    Uses SingleTurnSample + single_turn_score (synchronous) — no API keys required.
    retrieved_contexts (what RAG retrieved) is compared against reference_contexts
    (the PubMedQA golden reference contexts).

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
            Expected columns: question, generated_answer, retrieved_contexts,
            golden_contexts, golden_answer.
        metric: A RAGAS metric instance.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).
        is_old_metric_type: Boolean to indicate whether the metric is from old
        collections package

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def _run_ragas_llm_slice(eval_df_slice, api_key, model, metric_cls, embeddings, delay, key_idx):
    """Evaluate a contiguous slice of rows with an LLM-based RAGAS metric.

    Called in its own thread by evaluate_ragas_parallel. Each thread creates its own
    IntructorLLM + metric instance and its own asyncio event loop, so threads
    never share state and asyncio.run() never conflicts with Jupyter's main loop.

    Args:
        eval_df_slice: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: LLM-based RAGAS metric class (not an instance), e.g. ContextRecall.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of per-row scores (float or None on error) in slice order.
    """
    client = instructor.from_groq(
        AsyncGroq(api_key=api_key),
        mode=instructor.Mode.JSON,
    )

    ragas_llm = InstructorLLM(
        client=client,
        provider='groq',
        model=model,
        is_async=True,
    )
    metric = metric_cls(llm=llm, embeddings=embeddings)
    scores = []

    for row_idx, (_, row) in enumerate(eval_df_slice.iterrows()):
        try:
            sample = SingleTurnSample(
                user_input=row["question"],
                retrieved_contexts=row["retrieved_contexts"],
                reference_contexts=row["golden_contexts"],
                reference=row["golden_answer"],
                response=row["generated_answer"],
            )
            score = metric.single_turn_score(sample)
            scores.append(score)
        except Exception as e:
            print(f"[Key {key_idx}] Error on row {row_idx}: {e}")
            scores.append(None)

        if row_idx < len(eval_df_slice) - 1:
            time.sleep(delay)

    completed = sum(1 for s in scores if s is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(eval_df_slice)} rows scored")
    return scores


def evaluate_ragas_parallel(eval_df, metric_cls, key_rotator, embeddings, results_file=None, delay=None, rows_per_key=None):
    """Evaluate an LLM-based RAGAS metric in parallel, one API key per slice.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next slice, etc.
    Returns the same (scores, avg, scores_df) tuple as evaluate_ragas, so the
    result plugs directly into build_ragas_combined as a drop-in replacement.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        metric_cls: LLM-based RAGAS metric class (not an instance), e.g. ContextRecall.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).
        delay: Seconds between rows within each slice (default config.PARALLEL_DELAY_SECONDS).
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_name = metric_cls.__name__

    if len(eval_df) == 0:
        raise ValueError("eval_df is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(eval_df) > total_capacity:
        print(
            f"Warning: {len(eval_df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        eval_df = eval_df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(eval_df):
            break
        slices.append((key, i, eval_df.iloc[start: start + rows_per_key]))

    print(f"\n{len(eval_df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_scores = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_ragas_llm_slice,
                s, key, key_rotator.model, metric_cls, embeddings, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_scores[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_scores[idx] = [None] * len(s)

    all_scores = []
    for slice_scores in ordered_scores:
        all_scores.extend(slice_scores)

    valid = [s for s in all_scores if s is not None]
    avg = sum(valid) / len(valid) if valid else 0.0
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(valid)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df

def build_ragas_combined(eval_df, score_dfs, results_file=None):
    """Combine eval_df with per-metric score DataFrames into one summary CSV.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        score_dfs: List of score DataFrames from evaluate_ragas, each with
            question_index + one metric score column.
        results_file: Optional CSV path to save the combined DataFrame.

    Returns:
        Combined DataFrame with question_index, question, retrieved_contexts,
        golden_contexts, golden_answer, generated_response, and one column per metric.
    """
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx):
    """Evaluate a contiguous slice of test cases using a single dedicated API key.

    Called in its own thread by evaluate_deepeval_parallel. Test cases are evaluated
    sequentially with `delay` seconds between each to stay within the key's daily quota.

    Args:
        test_case_slice: List of LLMTestCase objects assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        threshold: Pass/fail threshold for the metric.
        delay: Seconds to sleep between test cases.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of deepeval test results in slice order.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    results = []

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm))
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config= DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None):
    """Assign a contiguous slice of test cases to each API key and run all slices in parallel.

    Key 0 gets test_cases[0:rows_per_key], key 1 gets the next slice, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    Args:
        test_cases: List of LLMTestCase objects built by build_test_cases.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        threshold: Pass/fail threshold for the metric (default 0.5).
        results_file: Optional CSV path to save per-sample scores.
        delay: Seconds between cases within each slice (defaults to config.DEEPEVAL_DELAY_SECONDS).
        rows_per_key: Max cases assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        List of deepeval test results in original case order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if not test_cases:
        raise ValueError("test_cases is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(
            f"Warning: {len(test_cases)} cases exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} cases will be processed."
        )
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        if results_file:
            rows = []
            for r in all_results:
                rows.append({
                    "question": r.input,
                    "generated_answer": r.actual_output,
                    "retrieved_contexts": r.retrieval_context,
                    "golden_answer": r.expected_output,
                    r.metrics_data[0].name: r.metrics_data[0].score,
                })
            pd.DataFrame(rows).to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results

---
## Setup

In [33]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 11 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260504_132359


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.
Generate this file once by running `python3 sampling.py` from the `research/` directory.
Keeping the split fixed is critical — regenerating mid-experiment would change which
questions each RAG variant sees, invalidating cross-experiment comparisons.

In [13]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
print(f"Structure of golden dataset")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(type(golden_df['golden_contexts'].iloc[0]))
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
Structure of golden dataset
<class 'list'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store

Load the pre-ingested vector store. Swap `load_chroma` for `load_faiss`, `load_qdrant`,
or `load_lancedb` to evaluate a different backend — everything downstream stays the same.

In [14]:
# Switch to load_faiss / load_qdrant / load_lancedb to evaluate a different store
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb


## Build Retrievers

In [15]:
cosine_retriever = get_cosine_retriever(vector_store, k=3)

## Run Naive RAG (Cosine Retrieval)

In [36]:
eval_dataset = run_rag_parallel(cosine_retriever, golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"naive_rag_{embedding_key}_chroma_cosine_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 2] Done — 20/20 rows collected
[Key 4] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 9] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected

Completed 200/200 questions total
Generated 200 answers


---
## RAGAS Evaluation

In [37]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.1804 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_minilm_context_recall_20260504_132359.csv


In [38]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.2979 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_minilm_context_precision_20260504_132359.csv


In [48]:
ragas_blue_scores, ragas_blue_avg, ragas_blue_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1853 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_minilm_bleu_20260504_132359.csv


In [52]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.3098 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_minilm_rouge_20260504_132359.csv


In [56]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_cr_df, ragas_cp_df, ragas_blue_df, ragas_rouge_df], results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_combined_{timestamp}.csv"))

Saved combined RAGAS results to /content/results/ragas/naive_rag_minilm_combined_20260504_132359.csv


## DeepEval Evaluation

In [25]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 11 API key(s)


In [51]:
deepeval_cr = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)

[Key 2] Error on case 1/20: 'NoneType' object has no attribute 'save'
[Key 7] Error on case 1/20: 'NoneType' object has no attribute 'save'


⚠ WARNING: No hyperparameters logged.
» ]8;id=551969;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.84s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=415324;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.71s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

✓ Evaluation completed 🎉! (time taken: 1.98s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 20/20 cases evaluated


⚠ WARNING: No hyperparameters logged.
» ]8;id=571272;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.78s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 9] Done — 20/20 cases evaluated

=== Contextual Recall: 0.9061 (avg over 198 samples) ===
Saved to /content/results/deepeval/naive_rag_minilm_ctx_recall_20260504_132359.csv


In [54]:
deepeval_cp = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)

[Key 8] Error on case 1/20: 'NoneType' object has no attribute 'save'


Warning: Could not load test run from disk: Expecting value: line 1 column 1 (char 0)

[Key 0] Error on case 1/20: 'NoneType' object has no attribute 'save'


Warning: Could not load test run from disk: Expecting value: line 1 column 1 (char 0)

Warning: Could not load test run from disk: Expecting value: line 1 column 1 (char 0)

...

✓ Evaluation completed 🎉! (time taken: 5.55s | token cost: None)
» Test Results (7 total tests):
   » Pass Rate: 85.71% | Passed: 6 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 19/20 cases evaluated


⚠ WARNING: No hyperparameters logged.
» ]8;id=53006;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.03s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 4] Done — 20/20 cases evaluated

=== Contextual Precision: 0.9590 (avg over 197 samples) ===
Saved to /content/results/deepeval/naive_rag_minilm_ctx_precision_20260504_132359.csv


In [55]:
deepeval_f = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



⚠ WARNING: No hyperparameters logged.
» ]8;id=778343;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.3s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

⚠ WARNING: No hyperparameters logged.
» ]8;id=421508;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.43s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

⚠ WARNING: No hyperparameters logged.
» ]8;id=930165;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 6.12s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 20/20 cases evaluated

=== Faithfulness: 0.9859 (avg over 200 samples) ===
Saved to /content/results/deepeval/naive_rag_minilm_faithfulness_20260504_132359.csv


In [28]:
from deepeval.metrics import AnswerRelevancyMetric

deepeval_cr = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_{embedding_key}_ans_relevancy_{timestamp}.csv")
)


165 cases split across 11 key(s) (15 cases/key max):
  Key 0: cases 0–14 (15 cases)
  Key 1: cases 15–29 (15 cases)
  Key 2: cases 30–44 (15 cases)
  Key 3: cases 45–59 (15 cases)
  Key 4: cases 60–74 (15 cases)
  Key 5: cases 75–89 (15 cases)
  Key 6: cases 90–104 (15 cases)
  Key 7: cases 105–119 (15 cases)
  Key 8: cases 120–134 (15 cases)
  Key 9: cases 135–149 (15 cases)
  Key 10: cases 150–164 (15 cases)

[Key 6] Error on case 1/15: 'NoneType' object has no attribute 'save'
[Key 4] Error on case 1/15: 'NoneType' object has no attribute 'save'


⚠ WARNING: No hyperparameters logged.
» ]8;id=802472;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.54s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=80519;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.16s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 10] Done — 15/15 cases evaluated

=== Answer Relevancy: 0.9781 (avg over 163 samples) ===
Saved to /content/results/deepeval/naive_rag_minilm_ans_relevancy_20260504_132359.csv
